# Test HKJC Football DataProvider

## 0. Import Packages

In [2]:
# Import Libraries
## Standard Libraries
import datetime
import json
import logging
import os
import sys

## Third-Party Libraries
import pandas as pd

## Local Libraries
sys.path.append("D:\\workspace\\hkjc_football")
from hkjc_core.data.data_provider import HKJC_Football_DataProvider
from hkjc_core.data.data_parser.match_result_data_parser import MatchResult_DataParser


In [3]:
hkjc_fb_dp = HKJC_Football_DataProvider(
    instance_id=42,
    log_level=logging.DEBUG,
    base_url="https://info.cld.hkjc.com/graphql/base/",
    max_retries=3,
    retry_backoff_factor=0.3,
)

## 1. Test Query for Team IDs

In [ ]:
team_id_dict = hkjc_fb_dp.get_team_ids()
team_info_df = pd.DataFrame(team_id_dict)
print(len(team_info_df))
team_info_df.head()


In [ ]:
team_id_dict

In [ ]:
team_info_df.loc[team_info_df["name_en"] == "Real Madrid"]

In [ ]:
team_info_df.to_csv("D:\\workspace\\hkjc_football\\sample_data\\team_data\\team_infos_table.csv", index=False)

## 2. Test Query for Historical Matches

In [ ]:
match_records_list = hkjc_fb_dp.get_historical_matches(
    start_date=datetime.date(2026, 1, 1),
    end_date=datetime.date(2026, 6, 12),
    team_id="50002589",
)


In [ ]:
len(match_records_list)

In [ ]:
match_records_list[0]

In [ ]:
match_records_list[0]["id"], match_records_list[1]["id"]

In [ ]:
with open("../../../../sample_data/match_data/match_records_real_madrid_sample_20260612.json", "w") as f:
    f.write(json.dumps(match_records_list))

## 3. Test Query for Match Results

In [4]:
match_results = hkjc_fb_dp.get_match_results(
    start_date=datetime.date(2026, 1, 1),
    end_date=datetime.date(2026, 6, 12),
    team_id="50002589",
)
match_results

[{'id': '50060631',
  'status': 'MATCHENDED',
  'frontEndId': 'FB2757',
  'matchDayOfWeek': '',
  'matchNumber': '',
  'matchDate': '2026-01-29+08:00',
  'kickOffTime': '2026-01-29T04:00:00.000+08:00',
  'sequence': '1.1769630400.11.UE Champions                                                                                                                    .1.08.Benfica                                                                                                                         .000',
  'homeTeam': {'id': '50000449', 'name_en': 'Benfica', 'name_ch': '賓菲加'},
  'awayTeam': {'id': '50002589', 'name_en': 'Real Madrid', 'name_ch': '皇家馬德里'},
  'tournament': {'code': 'UCL',
   'name_en': 'UE Champions',
   'name_ch': '歐洲聯賽冠軍盃'},
  'results': [{'homeResult': 0,
    'awayResult': 1,
    'ttlCornerResult': -1,
    'resultConfirmType': 5,
    'payoutConfirmed': True,
    'stageId': 2,
    'resultType': 1,
    'sequence': 45},
   {'homeResult': 1,
    'awayResult': 1,
    'ttlCornerRes

In [6]:
match_result = match_results[1]["results"]
print(match_result)

[{'homeResult': 0, 'awayResult': 0, 'ttlCornerResult': -1, 'resultConfirmType': 5, 'payoutConfirmed': True, 'stageId': 3, 'resultType': 1, 'sequence': 194}, {'homeResult': 0, 'awayResult': 1, 'ttlCornerResult': -1, 'resultConfirmType': 5, 'payoutConfirmed': True, 'stageId': 4, 'resultType': 1, 'sequence': 195}, {'homeResult': 0, 'awayResult': 2, 'ttlCornerResult': -1, 'resultConfirmType': 2, 'payoutConfirmed': True, 'stageId': 4, 'resultType': 1, 'sequence': 196}, {'homeResult': 0, 'awayResult': 2, 'ttlCornerResult': -1, 'resultConfirmType': 5, 'payoutConfirmed': True, 'stageId': 5, 'resultType': 1, 'sequence': 197}, {'homeResult': 1, 'awayResult': 0, 'ttlCornerResult': -1, 'resultConfirmType': 5, 'payoutConfirmed': True, 'stageId': 2, 'resultType': 2, 'sequence': 198}, {'homeResult': 2, 'awayResult': 0, 'ttlCornerResult': -1, 'resultConfirmType': 5, 'payoutConfirmed': True, 'stageId': 2, 'resultType': 2, 'sequence': 199}, {'homeResult': 3, 'awayResult': 0, 'ttlCornerResult': -1, 'resu

In [9]:
parser = MatchResult_DataParser()
# parse the result
parser.extract_goals(results=match_result, time_period="FT")

{'home': 0, 'away': 2, 'total': 2, 'display': '0-2'}

In [ ]:
with open("../../../../sample_data/match_data/match_results_sample_20260612.json", "w") as f:
    f.write(json.dumps(match_results))

## 4. Test Query for Match Odds

In [ ]:
match_odds = hkjc_fb_dp.get_match_odds_in_batch(
    match_id_list=['50060631', '50060535'],
)
len(match_odds)

In [ ]:
match_odds[1].keys()

In [ ]:
with open("../../../../sample_data/match_data/match_odds_sample_20260612.json", "w") as f:
    f.write(json.dumps(match_odds, indent=4))

## 5. Test Query for All Live Matches

In [ ]:
all_live_matches = hkjc_fb_dp.get_all_live_matches()
print(len(all_live_matches))


In [ ]:
all_live_matches[0]

In [ ]:
all_live_matches[0].keys()

In [ ]:
for match in all_live_matches:
    print(f"{match['homeTeam']['name_ch']} v.s. {match['awayTeam']['name_ch']}")

In [ ]:
with open("../../../../sample_data/match_data/all_live_matches_sample_20260612.json", "w") as f:
    f.write(json.dumps(all_live_matches, indent=4))

## 5. Test Query for Live Match Data

In [ ]:
live_match_data = hkjc_fb_dp.get_live_matches(
    match_id_list=["50069886"]
)
live_match_data

In [ ]:
with open("../../../../sample_data/match_data/live_match_data_sample_20260612.json", "w") as f:
    f.write(json.dumps(live_match_data, indent=4))